# Nhận diện & ĐẾM nhà ngập — Colab chạy 1 lần từ đầu đến cuối

**Không cần sửa gì trong notebook.** Bấm `Runtime > Run all` là chạy hết.

## Chuẩn bị (1 lần)
1. `Runtime > Change runtime type` → Hardware accelerator = **GPU** (chọn **A100** nếu có).
2. Dữ liệu: notebook tự tìm theo thứ tự sau, cái nào có trước thì dùng:
   - **(a)** đã có sẵn ở `/content/data`
   - **(b)** Colab Secret tên `ROBEFLOW_API_KEY` → tải đúng *merged_dataset version 1* (2.778 ảnh 640×640)
   - **(c)** file zip trên Google Drive: `MyDrive/nha_ngap_640.zip` (hoặc thư mục đã giải nén sẵn)
   - **(d)** hỏi bạn upload file zip từ máy
3. Nếu muốn dùng (b): mở biểu tượng 🔑 ở thanh trái Colab → Add new secret → name `ROBEFLOW_API_KEY`, value = API key Roboflow của bạn. Secret không bao giờ hiện trong code.

## Kết quả nằm ở đâu
- Mọi thứ lưu vào `MyDrive/nha_ngap_v002_out/` (checkpoint, log, bảng điểm, ảnh demo).
- **Đứt kết nối giữa chừng** → chạy lại `Run all`, notebook tự **resume** từ checkpoint trên Drive, không train lại từ đầu.

## Cấu hình đã chốt
| Tầng | Model |
|---|---|
| ⓐ Phát hiện nhà (đếm) | Cascade R-CNN + ResNet-50-FPN (mmdetection), 1 lớp `house` |
| ⓑ Phân loại | ConvNeXt-Tiny, 3 lớp Flooded / Not-Flooded / Partially-Flooded |

Dataset: `merged_dataset` v1 — 2.778 ảnh 640×640, 44.967 box. **CẤM YOLO.**


In [ ]:
# ============================================================================
# Ô 1/11 — Kiểm tra môi trường (Python / torch / GPU)
# ============================================================================
import sys, os, torch

print('Python ', sys.version.split()[0])
print('torch  ', torch.__version__, '| CUDA', torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit('CHUA BAT GPU -> Runtime > Change runtime type > Hardware accelerator = GPU (chon A100)')
p = torch.cuda.get_device_properties(0)
print('GPU    ', p.name, '| VRAM %.1f GB' % (p.total_memory / 1e9))


In [ ]:
# ============================================================================
# Ô 2/11 — Cai mmcv / mmengine / mmdet (tu dong do wheel dung voi torch hien tai)
#   Colab chay Python 3.13 -> OpenMMLab khong phat hanh wheel chinh thuc nua,
#   phai lay tu index cong dong miropsota. O nay tu tim dung wheel.
#   Dong thoi va 2 loi da biet: mmengine trung ten Adafactor, mmdet ghim mmcv <= 2.2.0
# ============================================================================
import os, re, sys, subprocess, urllib.request, urllib.parse, torch

IDX = 'https://miropsota.github.io/torch_packages_builder'
MARK = '/content/.mmdet_ready'


def sh(cmd):
    print('>>', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)


def tim_wheel_mmcv():
    py = 'cp%d%d' % sys.version_info[:2]
    tv = torch.__version__.split('+')[0]
    cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else ''
    hdh = 'linux_x86_64' if sys.platform.startswith('linux') else ('win_amd64' if os.name == 'nt' else 'macosx')
    html = urllib.request.urlopen(IDX + '/mmcv/', timeout=60).read().decode('utf-8', 'ignore')
    hrefs = re.findall(r'href="([^"]+\.whl)"', html)
    khop, tatca = [], []
    for h in hrefs:
        ten = urllib.parse.unquote(h.split('/')[-1])
        tatca.append(ten)
        if py in ten and ('pt' + tv) in ten and hdh in ten and (cu in ten if cu else True):
            khop.append(urllib.parse.urljoin(IDX + '/mmcv/', h))
    return khop, tatca, py, tv, cu


def va_mmengine():
    import importlib.util
    spec = importlib.util.find_spec('mmengine')
    if spec is None:
        return 'mmengine: CHUA CAI'
    root = os.path.join(os.path.dirname(spec.origin), 'optim')
    helper = (
        "def _va_reg(*a, **kw):\n"
        "    _ten = kw.get('name', a[0] if len(a) > 0 else None)\n"
        "    _force = kw.get('force', a[1] if len(a) > 1 else False)\n"
        "    _mod = kw.get('module', a[2] if len(a) > 2 else None)\n"
        "    if _mod is None:\n"
        "        def _d(_cls):\n"
        "            _va_reg(_ten, _force, _cls)\n"
        "            return _cls\n"
        "        return _d\n"
        "    if _ten is None:\n"
        "        _ten = getattr(_mod, '__name__', None)\n"
        "    try:\n"
        "        if not _force and _ten is not None and _ten in OPTIMIZERS.module_dict:\n"
        "            return None\n"
        "    except Exception:\n"
        "        pass\n"
        "    return OPTIMIZERS.register_module(_ten, _force, _mod)\n"
        "\n"
        "\n"
    )
    n, loi = 0, []
    for dirpath, _dn, files in os.walk(root):
        for fn in files:
            if not fn.endswith('.py'):
                continue
            p = os.path.join(dirpath, fn)
            s = open(p, encoding='utf-8').read()
            if 'OPTIMIZERS.register_module(' not in s and 'def _va_reg(' not in s:
                continue
            # Bỏ helper cũ (nếu phiên trước đã vá bằng bản hỏng) để thay bằng bản mới.
            # Không có bước này thì lần chạy sau thấy "_va_reg" có sẵn -> bỏ qua file,
            # helper hỏng cũ ở lại và lỗi Adafactor quay lại y nguyên.
            s = re.sub(r'^def _va_reg\(.*?^    return OPTIMIZERS\.register_module\([^\n]*\n',
                       '', s, flags=re.S | re.M)
            if 'def _va_reg(' in s:
                loi.append(fn)
                continue
            s = s.replace('OPTIMIZERS.register_module(', '_va_reg(')
            # Chèn helper ở ĐẦU file (sau dòng __future__ nếu có).
            # KHÔNG bám vào "^def " nữa: zero_optimizer.py chỉ có
            # "@OPTIMIZERS.register_module()" trên "class", không hề có ^def,
            # mà chèn ngay trước "class" thì chui vào giữa decorator -> SyntaxError.
            # Đầu file luôn an toàn vì helper chỉ tra OPTIMIZERS lúc ĐƯỢC GỌI.
            moc = None
            for mm2 in re.finditer(r'^from __future__ import .*$', s, re.M):
                moc = mm2
            vitri = (moc.end() + 1) if moc else 0
            s2 = s[:vitri] + helper + s[vitri:]
            try:
                compile(s2, p, 'exec')
            except SyntaxError:
                loi.append(fn)
                continue
            open(p, 'w', encoding='utf-8').write(s2)
            n += 1
    tb = 'mmengine: da va %d file (chan loi Adafactor trung ten)' % n
    if loi:
        tb += ' | BO QUA (loi cu phap): ' + ', '.join(loi)
    return tb


def va_mmdet():
    import importlib.util
    spec = importlib.util.find_spec('mmdet')
    if spec is None:
        return 'mmdet: CHUA CAI'
    p = os.path.join(os.path.dirname(spec.origin), '__init__.py')
    s = open(p, encoding='utf-8').read()
    if "mmcv_maximum_version = '2.3.0'" in s:
        return 'mmdet: da va truoc do'
    s2 = re.sub(r"mmcv_maximum_version\s*=\s*'2\.2\.0'", "mmcv_maximum_version = '2.3.0'", s)
    if s2 == s:
        return 'mmdet: KHONG thay dong can va (bao lai de kiem tra)'
    open(p, 'w', encoding='utf-8').write(s2)
    return 'mmdet: da noi mmcv_maximum_version -> 2.3.0'


def va_torch_load():
    """torch >= 2.6 doi mac dinh torch.load(weights_only=True). Checkpoint mmengine
    luu kem object HistoryBuffer -> bi chan -> UnpicklingError khi init_detector /
    test / resume. Moi checkpoint o day deu la do chinh minh train ra (hoac
    torchvision chinh thuc) nen mo lai weights_only=False, dung nhu truoc torch 2.6."""
    if getattr(torch.load, '_da_va', False):
        return 'torch.load: da vá truoc do'
    goc = torch.load

    def load_mo(*a, **kw):
        kw.setdefault('weights_only', False)
        return goc(*a, **kw)

    load_mo._da_va = True
    torch.load = load_mo
    return 'torch.load: da mo weights_only=False (torch %s)' % torch.__version__


if not os.path.exists(MARK):
    khop, tatca, py, tv, cu = tim_wheel_mmcv()
    print('Tim wheel mmcv khop: python %s | torch %s | %s -> %d wheel' % (py, tv, cu, len(khop)))
    for w in khop[:5]:
        print('   ', w)
    if not khop:
        print('--- 15 wheel co tren index (de bao loi) ---')
        for w in tatca[:15]:
            print('   ', w)
        raise SystemExit('DUNG LAI: khong co wheel mmcv khop voi runtime nay')
    sh('pip install -q "%s"' % khop[0])
    r = subprocess.run('pip install -q "mmdet==3.3.0" mmengine', shell=True)
    if r.returncode != 0:
        sh('pip install -q mmengine mmdet')
    open(MARK, 'w').write('ok')

print(va_mmengine())
print(va_mmdet())
print(va_torch_load())


In [ ]:
# ============================================================================
# Ô 3/11 — Kiem tra import + CUDA ops that su chay duoc
# ============================================================================
import torch, cv2
import mmcv, mmengine, mmdet
from mmcv.ops import nms

print('mmcv     ', mmcv.__version__)
print('mmengine ', mmengine.__version__)
print('mmdet    ', mmdet.__version__)
print('cv2      ', cv2.__version__)

b = torch.rand(200, 4).cuda() * 500
b[:, 2:] += b[:, :2] + 100
s = torch.rand(200).cuda()
keep, _ = nms(b, s, 0.5)
print('NMS tren GPU OK, giu lai', keep.shape[0], '/ 200 box')


In [ ]:
# ============================================================================
# Ô 4/11 — Lay du lieu (tu dong: co san -> Roboflow -> Drive -> upload)
# ============================================================================
import os, json, glob, zipfile, shutil, subprocess

DATA = '/content/data'
SPL = ['train', 'valid', 'test']
DRIVE_OUT = '/content/drive/MyDrive/nha_ngap_v002_out'


def _dem(p):
    try:
        return len(json.load(open(p, encoding='utf-8'))['images'])
    except Exception:
        return 0


def co_du_lieu():
    """Khong chi du 50 anh moi split — phai dung bo 2778. Neu phien truoc da vo
    nham bo khac vao /content/data thi lan chay nay phai tai lai."""
    n = [_dem('%s/%s/_annotations.coco.json' % (DATA, s)) for s in SPL]
    if any(x < 50 for x in n):
        return False
    return 2700 <= sum(n) <= 2900


def tim_split(root, sau=4):
    got = {}
    root = root.rstrip('/')
    for dp, dn, fn in os.walk(root):
        if dp.count(os.sep) - root.count(os.sep) > sau:
            dn[:] = []
            continue
        if '_annotations.coco.json' in fn:
            s = os.path.basename(dp)
            if s in SPL and s not in got:
                got[s] = dp
    return got


def nhan_vao(root):
    """CHI nhan dung bo 2778 anh. Ten thu muc train/valid/test giong nhau o rat
    nhieu bo dataset khac, nen phai dem anh truoc khi nhan."""
    got = tim_split(root)
    if len(got) < 3:
        return False
    n = dict((sp, _dem('%s/_annotations.coco.json' % src)) for sp, src in got.items())
    tong = sum(n.values())
    if not (2700 <= tong <= 2900):
        print('[BO QUA] %s -> %d anh %s, KHONG phai bo 2778' % (root, tong, n))
        return False
    os.makedirs(DATA, exist_ok=True)
    for s, src in got.items():
        dst = '%s/%s' % (DATA, s)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    print('[OK] da lay du lieu tu', root, n)
    return True


def thu_roboflow():
    try:
        from google.colab import userdata
        key = userdata.get('ROBEFLOW_API_KEY')
    except Exception:
        key = None
    if not key:
        print('[Roboflow] khong co secret ROBEFLOW_API_KEY -> bo qua')
        return False
    try:
        import roboflow  # noqa
    except Exception:
        print('[Roboflow] dang cai goi roboflow...')
        if subprocess.run('pip install -q roboflow', shell=True).returncode != 0:
            return False
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=key)
        prj = rf.workspace('housedetectionclassification-jugwz').project('merged_dataset-dk0ch')
        prj.version(1).download('coco', location='/content/rf_dl', overwrite=True)
        return nhan_vao('/content/rf_dl')
    except Exception as e:
        print('[Roboflow] loi:', e)
        return False


def mount_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('[Drive] khong mount duoc:', e)
    return os.path.isdir('/content/drive/MyDrive')


def dem_trong_zip(z):
    """Doc thang _annotations.coco.json BEN TRONG zip de dem anh — KHONG giai nen.
    Nho vay zip sai bi loai trong vai giay, thay vi giai nen ca GB roi moi biet."""
    n = {}
    try:
        with zipfile.ZipFile(z) as f:
            for ten in f.namelist():
                if not ten.endswith('/_annotations.coco.json'):
                    continue
                sp = ten[:-len('/_annotations.coco.json')].rsplit('/', 1)[-1]
                if sp in SPL and sp not in n:
                    try:
                        n[sp] = len(json.loads(f.read(ten).decode('utf-8'))['images'])
                    except Exception:
                        n[sp] = 0
    except Exception as e:
        print('[BO QUA] %s -> khong mo duoc zip (%s)' % (os.path.basename(z), e))
        return None
    if len(n) < 3:
        print('[BO QUA] %s -> thieu split (chi thay %s)' % (os.path.basename(z), sorted(n)))
        return None
    return n


def thu_zip(z, dst):
    """Nhan 1 file zip: dem truoc, chi giai nen khi dung bo 2778."""
    n = dem_trong_zip(z)
    if not n:
        return False
    tong = sum(n.values())
    if not (2700 <= tong <= 2900):
        print('[BO QUA] %s -> %d anh %s, KHONG phai bo 2778'
              % (os.path.basename(z), tong, n))
        return False
    print('[Zip] %s -> %d anh %s, DUNG BO -> giai nen' % (os.path.basename(z), tong, n))
    if os.path.exists(dst):
        shutil.rmtree(dst)
    with zipfile.ZipFile(z) as f:
        f.extractall(dst)
    return nhan_vao(dst)


def thu_drive():
    if not os.path.isdir('/content/drive/MyDrive') and not mount_drive():
        return False
    for r in ['/content/drive/MyDrive'] + glob.glob('/content/drive/MyDrive/*'):
        try:
            if nhan_vao(r):
                return True
        except Exception:
            pass
    zips = glob.glob('/content/drive/MyDrive/*.zip') + glob.glob('/content/drive/MyDrive/*/*.zip')
    # Uu tien zip co '2778' trong ten (bo chuan) len truoc cho log de doc
    zips = sorted(set(zips), key=lambda p: (0 if '2778' in os.path.basename(p) else 1, p))
    for k, z in enumerate(zips):
        if thu_zip(z, '/content/unz/z%d' % k):
            return True
    print('[Drive] khong thay du lieu trong MyDrive')
    return False


def thu_upload():
    try:
        from google.colab import files
    except Exception:
        return False
    print('[Upload] Chon file ZIP dataset 640 tu may cua ban...')
    up = files.upload()
    for k, name in enumerate(up):
        if name.lower().endswith('.zip'):
            if thu_zip('/content/' + name, '/content/unz/u%d' % k):
                return True
    return False


if co_du_lieu():
    print('Da co du lieu -> bo qua buoc tai')
else:
    if not (thu_roboflow() or thu_drive() or thu_upload()):
        raise SystemExit('KHONG LAY DUOC DU LIEU -> xem lai 3 cach o phan dau notebook')

if not os.path.isdir('/content/drive/MyDrive'):
    mount_drive()

tong = 0
for s in SPL:
    n = _dem('%s/%s/_annotations.coco.json' % (DATA, s))
    tong += n
    print('%-6s %5d anh' % (s, n))
print('TONG   %5d anh (dataset chuan = 2778)' % tong)
if not (2700 <= tong <= 2900):
    raise SystemExit('SAI DATASET: %d anh. Bo dung phai la 2778 anh '
                     '(train 2116 / valid 459 / test 203).' % tong)


In [ ]:
# ============================================================================
# Ô 5/11 — Thong ke du lieu + chuyen sang 2 dinh dang:
#   (1) ann_house/  -> 1 lop 'house' cho DETECTOR
#   (2) cls_*.json  -> 3 lop goc cho CLASSIFIER
# ============================================================================
import os, json, collections

ANN = '/content/data/ann_house'
os.makedirs(ANN, exist_ok=True)
# CO_DRIVE dinh nghia o DAY (o 5) de moi o sau (6/8/10) dung duoc, ke ca khi
# nguoi dung bo qua o 6 vi detector da train xong (loi NameError da gap that).
CO_DRIVE = os.path.isdir('/content/drive/MyDrive')
OUT = DRIVE_OUT if CO_DRIVE else '/content/nha_ngap_out'
os.makedirs(OUT, exist_ok=True)

LOG = []


def doc(p):
    return json.load(open(p, encoding='utf-8'))


ten_lop = {}
for s in SPL:
    for c in doc('%s/%s/_annotations.coco.json' % (DATA, s))['categories']:
        ten_lop[c['id']] = c['name']
LOG.append('Ten lop trong file COCO: %s' % ten_lop)

thong_ke = {}
for s in SPL:
    d = doc('%s/%s/_annotations.coco.json' % (DATA, s))
    id2fn = {im['id']: im['file_name'] for im in d['images']}
    dem = collections.Counter(a['category_id'] for a in d['annotations'])
    dung = sorted([cid for cid in dem if dem[cid] > 0])
    ten_dung = [ten_lop[cid] for cid in dung]
    LOG.append('')
    LOG.append('--- %s: %d anh, %d box ---' % (s, len(d['images']), len(d['annotations'])))
    for cid in dung:
        LOG.append('    %-20s %6d box' % (ten_lop[cid], dem[cid]))
    canh = sorted(max(a['bbox'][2], a['bbox'][3]) for a in d['annotations'])

    def pct(q):
        return canh[min(len(canh) - 1, int(q * len(canh)))]
    LOG.append('    canh dai box: nho nhat %.0f | 25%% %.0f | trung vi %.0f | 75%% %.0f | lon nhat %.0f'
               % (canh[0], pct(0.25), pct(0.5), pct(0.75), canh[-1]))
    LOG.append('    box < 32px: %.1f%%   |   box < 64px: %.1f%%'
               % (100.0 * sum(1 for c in canh if c < 32) / len(canh),
                  100.0 * sum(1 for c in canh if c < 64) / len(canh)))

    # --- detector: 1 lop house ---
    anns = []
    for k, a in enumerate(d['annotations']):
        b = dict(a)
        b['id'] = k
        b['category_id'] = 1
        b['area'] = float(a['bbox'][2]) * float(a['bbox'][3])
        b['iscrowd'] = 0
        anns.append(b)
    det = dict(images=d['images'], annotations=anns,
               categories=[dict(id=1, name='house', supercategory='none')],
               licenses=d.get('licenses', []), info=d.get('info', {}))
    json.dump(det, open('%s/%s.json' % (ANN, s), 'w', encoding='utf-8'))

    # --- classifier: 3 lop goc ---
    man = []
    for a in d['annotations']:
        man.append(dict(file_name=id2fn[a['image_id']], bbox=list(a['bbox']),
                        label=dung.index(a['category_id']), ten=ten_lop[a['category_id']]))
    json.dump(man, open('%s/cls_%s.json' % (ANN, s), 'w', encoding='utf-8'))
    thong_ke[s] = dict(ten=ten_dung, so_box=len(man))

TEN_LOP_CLS = thong_ke['train']['ten']
LOG.append('')
LOG.append('Lop cho CLASSIFIER (3 lop): %s' % TEN_LOP_CLS)
LOG.append('Lop cho DETECTOR (1 lop): house')

open(os.path.join(OUT, '01_thong_ke_du_lieu.txt'), 'w', encoding='utf-8').write('\n'.join(LOG))
print('\n'.join(LOG))


In [ ]:
# ============================================================================
# Ô 6/11 — TRAIN DETECTOR: Cascade R-CNN + ResNet-50-FPN (1 lop 'house')
#   - tu tim config goc trong mmdet roi ghi de
#   - luu work_dir tren Drive -> dut ket noi thi chay lai se RESUME
# ============================================================================
import os, glob, torch, mmdet
from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.runner.checkpoint import find_latest_checkpoint

CO_DRIVE = os.path.isdir('/content/drive/MyDrive')
WORK = (DRIVE_OUT + '/work_det') if CO_DRIVE else '/content/work_det'
os.makedirs(WORK, exist_ok=True)

# Config cua mmdet nam trong 'mmdet/.mim/configs/...' — thu muc .mim bat dau
# bang dau cham, ma glob.glob KHONG BAO GIO di vao thu muc an -> phai dung os.walk.
duong = []
for _dp, _dn, _fn in os.walk(os.path.dirname(mmdet.__file__)):
    if 'cascade-rcnn_r50_fpn_1x_coco.py' in _fn:
        duong.append(os.path.join(_dp, 'cascade-rcnn_r50_fpn_1x_coco.py'))
duong.sort()
if not duong:
    raise SystemExit('Khong tim thay config goc cascade-rcnn_r50_fpn_1x_coco.py trong mmdet')
print('config goc:', duong[0])
cfg = Config.fromfile(duong[0])

CLS = ('house',)
ROOT = '/content/data'
EPOCH = 20
BATCH = 2

for dl, s in ((cfg.train_dataloader, 'train'), (cfg.val_dataloader, 'valid'), (cfg.test_dataloader, 'test')):
    if dl.dataset.get('type') != 'CocoDataset':
        print('CANH BAO: dataset kieu', dl.dataset.get('type'), '-> van thu chay')
    dl.dataset.ann_file = 'ann_house/%s.json' % s
    dl.dataset.data_root = ROOT
    dl.dataset.data_prefix = dict(img='%s/' % s)
    dl.dataset.metainfo = dict(classes=CLS)

cfg.data_root = ROOT
cfg.metainfo = dict(classes=CLS)
# CocoMetric cua mmdet 3.3.0 KHONG nhan tham so 'metainfo' -> nhet vao la
# TypeError ngay trong Runner.build_evaluator. Ten lop CocoMetric tu lay tu
# dataset_meta['classes'] ma ValLoop/TestLoop gan truoc khi danh gia.
cfg.val_evaluator.ann_file = '%s/ann_house/valid.json' % ROOT
cfg.test_evaluator.ann_file = '%s/ann_house/test.json' % ROOT

# Config goc la cua COCO 80 lop -> 3 tang Cascade phai sua ve dung so lop cua minh.
for _h in cfg.model.roi_head.bbox_head:
    _h.num_classes = len(CLS)

cfg.train_dataloader.batch_size = BATCH
cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.batch_size = 1
cfg.test_dataloader.batch_size = 1
cfg.train_cfg = dict(type='EpochBasedTrainLoop', max_epochs=EPOCH, val_interval=2)
cfg.val_cfg = dict(type='ValLoop')
cfg.test_cfg = dict(type='TestLoop')

cfg.optim_wrapper.type = 'AmpOptimWrapper'
cfg.optim_wrapper.optimizer.lr = 0.01
cfg.optim_wrapper.accumulative_counts = 4
cfg.param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(type='MultiStepLR', begin=0, end=EPOCH, milestones=[14, 18], gamma=0.1, by_epoch=True),
]
cfg.default_hooks.checkpoint.interval = 2
cfg.default_hooks.checkpoint.max_keep_ckpts = 2
cfg.default_hooks.logger.interval = 50
cfg.work_dir = WORK
cfg.randomness = dict(seed=0, deterministic=False)
cfg.resume = False

W_COCO = ('https://download.openmmlab.com/mmdetection/v2.0/cascade_rcnn/'
          'cascade_rcnn_r50_fpn_1x_coco/cascade_rcnn_r50_fpn_1x_coco_20200316-3dc56deb.pth')
cfg.load_from = None
try:
    import urllib.request
    urllib.request.urlopen(W_COCO, timeout=30)
    cfg.load_from = W_COCO
    print('Se khoi tao tu weight COCO (tot hon la train tu dau)')
except Exception as e:
    print('Khong tai duoc weight COCO -> train tu dau:', e)

cfg.dump(os.path.join(WORK, 'cau_hinh_detector.py'))
print('epochs =', EPOCH, '| batch =', BATCH, '| work_dir =', WORK)
print('config da luu:', os.path.join(WORK, 'cau_hinh_detector.py'))

best_da_co = sorted(glob.glob(os.path.join(WORK, 'best_*.pth')))
if best_da_co:
    print('DA CO checkpoint tot nhat -> bo qua train:', best_da_co[-1])
else:
    # mmengine runner.py:1691 — resume=True ma load_from KHAC None thi no coi
    # load_from LA checkpoint de resume (chu khong phai de khoi tao) -> di resume
    # tu weight COCO 2020 (file do khong co 'message_hub') -> KeyError: 'message_hub'.
    # Nen khi resume thi PHAI bo load_from. Checkpoint do minh train thi luon co
    # 'message_hub' (runner.py:2233), va con giu duoc optimizer + lr scheduler.
    ck_resume = find_latest_checkpoint(WORK)
    if ck_resume is None:
        cfg.resume = False
        print('Chua co checkpoint -> train moi (khoi tao tu weight COCO)')
    else:
        cfg.resume = True
        cfg.load_from = None
        print('RESUME tu checkpoint gan nhat tren Drive:', ck_resume)
    Runner.from_cfg(cfg).train()
    print('TRAIN XONG')


In [ ]:
# ============================================================================
# Ô 7/11 — DANH GIA DETECTOR tren tap test (mAP, va mAP cho nha NHO)
# ============================================================================
import os, glob, json
from mmengine.config import Config
from mmengine.runner import Runner

best = sorted(glob.glob(os.path.join(WORK, 'best_*.pth'))) or sorted(glob.glob(os.path.join(WORK, 'epoch_*.pth')))
if not best:
    raise SystemExit('Chua co checkpoint -> cho o 6 chay xong')
CKPT_DET = best[-1]
print('dung checkpoint:', CKPT_DET)

cfg2 = Config.fromfile(os.path.join(WORK, 'cau_hinh_detector.py'))
cfg2.load_from = CKPT_DET
cfg2.resume = False
met = Runner.from_cfg(cfg2).test()

# Runner.test() cua mmengine 0.10.7 tra ve DICT (runner.py:1804: "dict: A dict of
# metrics on testing set"), KHONG phai list. Ban cu lam 'for m in met' nen m la mot
# KEY (chuoi) -> sorted(m) duyet tung KY TU -> m[k] => TypeError.
# Chuan hoa ve list truoc khi in, va in ca gia tri khong phai so (vi du chuoi
# 'bbox_mAP_copypaste' = '0.277 0.584 ...').
MET = [met] if isinstance(met, dict) else list(met)
L = ['=== KET QUA DETECTOR (tap test) ===', 'checkpoint: %s' % CKPT_DET, '']
for m in MET:
    for k in sorted(m):
        v = m[k]
        if isinstance(v, (int, float)):
            v = '%.4f' % v
        L.append('%-28s %s' % (k, v))
    L.append('')
    L.append('GHI CHU: bbox_mAP_s = do chinh xac voi nha NHO (<32px) - day la chi so quan trong nhat')
    L.append('        voi yeu cau dem nha nho / nha o xa cua do an.')
open(os.path.join(OUT, '02_ket_qua_detector.txt'), 'w', encoding='utf-8').write('\n'.join(L))
print('\n'.join(L))


In [ ]:
# ============================================================================
# Ô 8/11 — TRAIN CLASSIFIER: ConvNeXt-Tiny 3 lop tren crop nha (box that)
# ============================================================================
import os, json, glob, random, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
dev = 'cuda'

WORK_CLS = (DRIVE_OUT + '/work_cls') if CO_DRIVE else '/content/work_cls'
os.makedirs(WORK_CLS, exist_ok=True)
SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
TF = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class AnhNha(Dataset):
    def __init__(self, split, train=False, margin=0.15):
        self.items = json.load(open('%s/cls_%s.json' % (ANN, split), encoding='utf-8'))
        self.dir = '%s/%s' % (DATA, split)
        self.train = train
        self.margin = margin

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        im = Image.open(os.path.join(self.dir, it['file_name'])).convert('RGB')
        W, H = im.size
        x, y, w, h = it['bbox']
        if self.train:
            x += random.uniform(-0.2, 0.2) * w
            y += random.uniform(-0.2, 0.2) * h
            s = random.uniform(0.8, 1.3)
            cx, cy = x + w / 2.0, y + h / 2.0
            w, h = w * s, h * s
            x, y = cx - w / 2.0, cy - h / 2.0
        m = self.margin
        x, y, w, h = x - w * m, y - h * m, w * (1 + 2 * m), h * (1 + 2 * m)
        x0, y0 = max(0, int(x)), max(0, int(y))
        x1, y1 = min(W, int(x + w + 0.5)), min(H, int(y + h + 0.5))
        if x1 - x0 < 4 or y1 - y0 < 4:
            x0, y0, x1, y1 = 0, 0, W, H
        c = im.crop((x0, y0, x1, y1)).resize((SIZE, SIZE), Image.BILINEAR)
        if self.train and random.random() < 0.5:
            c = c.transpose(Image.FLIP_LEFT_RIGHT)
        return TF(c), it['label']


ds_tr = AnhNha('train', train=True)
ds_va = AnhNha('valid')
print('crop train:', len(ds_tr), '| crop valid:', len(ds_va))

dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True, num_workers=4, drop_last=True)
dl_va = DataLoader(ds_va, batch_size=128, shuffle=False, num_workers=4)

model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(TEN_LOP_CLS))
model = model.to(dev)

EP = 12
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
scaler = torch.cuda.amp.GradScaler()
crit = nn.CrossEntropyLoss(label_smoothing=0.05)

CK_CLS = os.path.join(WORK_CLS, 'convnext_tiny_3lop.pth')
if os.path.exists(CK_CLS):
    print('DA CO checkpoint classifier -> bo qua train')
else:
    tot = len(dl_tr)
    for ep in range(EP):
        model.train()
        tong, dung, mat = 0, 0, 0.0
        for it, (xb, yb) in enumerate(dl_tr):
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                out = model(xb)
                loss = crit(out, yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            mat += loss.item() * xb.size(0)
            dung += (out.argmax(1) == yb).sum().item()
            tong += xb.size(0)
            if (it + 1) % 200 == 0:
                print('  ep %2d  it %4d/%d  loss %.4f  acc %.3f' % (ep + 1, it + 1, tot, mat / tong, dung / tong))
        sch.step()
        print('EP %2d xong | loss %.4f | acc train %.3f' % (ep + 1, mat / tong, dung / tong))
    torch.save({'model': model.state_dict(), 'ten_lop': TEN_LOP_CLS}, CK_CLS)
    print('da luu', CK_CLS)

# ---- danh gia tren valid ----
ck = torch.load(CK_CLS, map_location=dev)
model.load_state_dict(ck['model'])
model.eval()
NC = len(TEN_LOP_CLS)
cm = np.zeros((NC, NC), dtype=int)
with torch.no_grad():
    for xb, yb in dl_va:
        pr = model(xb.to(dev)).argmax(1).cpu().numpy()
        for t, p in zip(yb.numpy(), pr):
            cm[t, p] += 1

L = ['=== KET QUA CLASSIFIER ConvNeXt-Tiny (crop valid) ===', '']
L.append('confusion matrix (dong = nhan dung, cot = model doan):')
L.append('%-22s' % '' + ''.join('%14s' % n for n in TEN_LOP_CLS))
for i in range(NC):
    L.append('%-22s' % TEN_LOP_CLS[i] + ''.join('%14d' % cm[i, j] for j in range(NC)))
L.append('')
for i, n in enumerate(TEN_LOP_CLS):
    tong_i = cm[i].sum()
    L.append('%-22s acc %.3f  (%d/%d)' % (n, cm[i, i] / max(tong_i, 1), cm[i, i], tong_i))
L.append('')
L.append('acc toan bo: %.4f' % (np.trace(cm) / max(cm.sum(), 1)))
a = TEN_LOP_CLS.index('Flooded')
b = TEN_LOP_CLS.index('Partially-Flooded')
hb = cm[a, a] + cm[b, b]
tb = cm[a].sum() + cm[b].sum()
L.append('rieng cap kho nhat Flooded vs Partially-Flooded: %.3f (%d/%d)' % (hb / max(tb, 1), hb, tb))
open(os.path.join(OUT, '03_ket_qua_classifier.txt'), 'w', encoding='utf-8').write('\n'.join(L))
print('\n'.join(L))


In [ ]:
# ============================================================================
# Ô 9/11 — CHAY THAT: phat hien -> phan loai -> DEM nha tren toan bo tap test
# ============================================================================
import os, json, glob, collections, numpy as np, torch
from PIL import Image, ImageDraw
from torchvision import transforms
import torch.nn as nn
from torchvision.models import convnext_tiny
from mmdet.apis import init_detector, inference_detector

THR = 0.30

# O 9 tu tim lai duong dan neu o 6 / o 7 khong chay trong phien nay — checkpoint
# da nam san tren Drive. Chi doc duong dan, KHONG train lai, KHONG dung toi nhan.
if 'WORK' not in globals():
    WORK = (DRIVE_OUT + '/work_det') if CO_DRIVE else '/content/work_det'
if 'CKPT_DET' not in globals():
    _b = sorted(glob.glob(os.path.join(WORK, 'best_*.pth'))) or \
         sorted(glob.glob(os.path.join(WORK, 'epoch_*.pth')))
    CKPT_DET = _b[-1] if _b else None
if not CKPT_DET or not os.path.exists(CKPT_DET):
    raise SystemExit('Khong thay checkpoint detector trong %s -> phai chay o 6 truoc.' % WORK)

model_det = init_detector(os.path.join(WORK, 'cau_hinh_detector.py'), CKPT_DET, device='cuda:0')
ck = torch.load(CK_CLS, map_location='cuda')
model_cls = convnext_tiny()
model_cls.classifier[2] = nn.Linear(model_cls.classifier[2].in_features, len(TEN_LOP_CLS))
model_cls.load_state_dict(ck['model'])
model_cls = model_cls.cuda().eval()
TF2 = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

d_test = json.load(open('%s/test/_annotations.coco.json' % DATA, encoding='utf-8'))
id2fn = {im['id']: im['file_name'] for im in d_test['images']}
ten = {c['id']: c['name'] for c in d_test['categories']}
gt = collections.defaultdict(lambda: collections.Counter())
for a in d_test['annotations']:
    gt[id2fn[a['image_id']]][ten[a['category_id']]] += 1


def phan_loai_crop(im, boxes):
    """boxes = pred_instances.bboxes cua mmdet = (x1, y1, x2, y2).
    Crop phai GIONG HET luc train (o 8): nha them 0.15 moi ben, lam tron +0.5.
    (Ban cu doc xyxy thanh xywh -> crop lech tam va to gap doi.)"""
    if len(boxes) == 0:
        return []
    W, H = im.size
    batch = []
    for x1, y1, x2, y2 in boxes:
        m = 0.15
        bw, bh = x2 - x1, y2 - y1
        x0, y0 = max(0, int(x1 - bw * m)), max(0, int(y1 - bh * m))
        x3, y3 = min(W, int(x2 + bw * m + 0.5)), min(H, int(y2 + bh * m + 0.5))
        if x3 - x0 < 4 or y3 - y0 < 4:
            x0, y0, x3, y3 = 0, 0, W, H
        batch.append(TF2(im.crop((x0, y0, x3, y3)).resize((SIZE, SIZE), Image.BILINEAR)))
    with torch.no_grad():
        out = model_cls(torch.stack(batch).cuda()).argmax(1).cpu().numpy()
    return [TEN_LOP_CLS[i] for i in out]


MAU = {'Flooded': (255, 40, 40), 'Partially-Flooded': (255, 165, 0), 'Not-Flooded': (40, 220, 40)}
ds_anh = sorted(d_test['images'], key=lambda im: -len(gt[im['file_name']]))
sai = collections.Counter()
tong_gt = collections.Counter()
tong_pr = collections.Counter()
dep = 0
for im_info in d_test['images']:
    fn = im_info['file_name']
    p = '%s/test/%s' % (DATA, fn)
    r = inference_detector(model_det, p)
    pi = r.pred_instances
    b = pi.bboxes.cpu().numpy()
    sc = pi.scores.cpu().numpy()
    b = b[sc >= THR]
    im = Image.open(p).convert('RGB')
    nhan = phan_loai_crop(im, b)
    dem = collections.Counter(nhan)
    for k in TEN_LOP_CLS:
        tong_gt[k] += gt[fn][k]
        tong_pr[k] += dem[k]
    if gt[fn]:
        dep += 1
        for k in TEN_LOP_CLS:
            sai[k] += abs(dem[k] - gt[fn][k])

L = ['=== DEM NHA TREN TOAN BO TAP TEST (%d anh) ===' % len(d_test['images']), '']
L.append('%-22s %10s %10s %10s' % ('lop', 'thuc te', 'model dem', 'lech TB/anh'))
for k in TEN_LOP_CLS:
    L.append('%-22s %10d %10d %10.2f' % (k, tong_gt[k], tong_pr[k], sai[k] / max(dep, 1)))
L.append('%-22s %10d %10d %10.2f' % ('TONG', sum(tong_gt.values()), sum(tong_pr.values()),
                                     sum(sai.values()) / max(dep, 1)))
L.append('')
L.append('(chi tinh tren %d anh co nha; %d anh khong nha khong tinh vao lech TB)' % (dep, len(d_test['images']) - dep))
open(os.path.join(OUT, '04_dem_nha_test.txt'), 'w', encoding='utf-8').write('\n'.join(L))
print('\n'.join(L))

# ---- anh demo: 6 anh nhieu nha nhat ----
for k, im_info in enumerate(ds_anh[:6]):
    fn = im_info['file_name']
    p = '%s/test/%s' % (DATA, fn)
    im = Image.open(p).convert('RGB')
    r = inference_detector(model_det, p)
    b = r.pred_instances.bboxes.cpu().numpy()
    sc = r.pred_instances.scores.cpu().numpy()
    b = b[sc >= THR]
    nhan = phan_loai_crop(im, b)
    dr = ImageDraw.Draw(im)
    for (x1, y1, x2, y2), n in zip(b, nhan):
        c = MAU.get(n, (255, 255, 255))
        dr.rectangle([x1, y1, x2, y2], outline=c, width=2)
    dr.rectangle([0, 0, 430, 22], fill=(0, 0, 0))
    dr.text((5, 5), 'model dem: %d nha | thuc te: %d' % (len(b), len(gt[fn])), fill=(255, 255, 0))
    im.save(os.path.join(OUT, 'demo_%02d.jpg' % (k + 1)))
print('da luu 6 anh demo vao', OUT)


In [ ]:
# ============================================================================
# Ô 10/11 — TONG KET: gom ket qua + duong dan file
# ============================================================================
import os, glob, shutil

print('THU MUC KET QUA:', OUT)
for f in sorted(glob.glob(os.path.join(OUT, '*.txt')) + glob.glob(os.path.join(OUT, 'demo_*.jpg'))):
    print('   %-42s %8.1f KB' % (os.path.basename(f), os.path.getsize(f) / 1024.0))
for f in sorted(glob.glob(os.path.join(WORK, '*.pth'))):
    print('   %-42s %8.1f MB' % (os.path.basename(f), os.path.getsize(f) / 1e6))
for f in sorted(glob.glob(os.path.join(WORK_CLS, '*.pth'))):
    print('   %-42s %8.1f MB' % (os.path.basename(f), os.path.getsize(f) / 1e6))

print('')
print('Tom tat:')
for f in ['01_thong_ke_du_lieu.txt', '02_ket_qua_detector.txt', '03_ket_qua_classifier.txt', '04_dem_nha_test.txt']:
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        d = open(p, encoding='utf-8').read().splitlines()
        print('---', f, '---')
        for line in d[:14]:
            print('   ', line)

if not CO_DRIVE:
    print('')
    print('!!! Chua mount Google Drive -> ket qua nam trong /content (se mat khi tat may).')
    print('    Tai ve ngay: Files (bieu tuong thu muc ben trai) > content > nha_ngap_out > Download')


In [ ]:
# ============================================================================
# Ô 11/11 — WEB APP (Gradio): dua anh flycam len -> nhan dien -> hien ket qua
#   Chay duoc o PHIEN MOI: chi can o 1, 2, 3 da chay xong (khong can tai lai
#   dataset, khong train lai) — 2 model nap thang tu checkpoint tren Drive.
# ============================================================================
import os, glob, collections, subprocess, sys, tempfile
import torch, torch.nn as nn
from PIL import Image, ImageDraw
from torchvision import transforms
from torchvision.models import convnext_tiny

try:
    import gradio as gr
except ImportError:
    print('dang cai gradio...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)
    import gradio as gr

from mmdet.apis import init_detector, inference_detector

# ---- tham so: GIU NGUYEN nhu o 8 / o 9, doi la ket qua sai ----
THR = 0.30          # nguong tin cay detector (gia tri mac dinh cua thanh truot)
SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
MAU = {'Flooded': (255, 40, 40), 'Partially-Flooded': (255, 165, 0), 'Not-Flooded': (40, 220, 40)}

# ---- tu tim duong dan: o 11 khong phu thuoc o 4 / 5 / 8 / 9 ----
DRIVE_OUT = '/content/drive/MyDrive/nha_ngap_v002_out'
if not os.path.isdir('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('[Drive] khong mount duoc:', e)
CO_DRIVE = os.path.isdir('/content/drive/MyDrive')
WORK = (DRIVE_OUT + '/work_det') if CO_DRIVE else '/content/work_det'
WORK_CLS = (DRIVE_OUT + '/work_cls') if CO_DRIVE else '/content/work_cls'

_ck = sorted(glob.glob(os.path.join(WORK, 'best_*.pth'))) or \
      sorted(glob.glob(os.path.join(WORK, 'epoch_*.pth')))
if not _ck:
    raise SystemExit('Khong thay checkpoint detector trong %s -> phai chay o 6 truoc.' % WORK)
CKPT_DET = _ck[-1]
CK_CLS = os.path.join(WORK_CLS, 'convnext_tiny_3lop.pth')
if not os.path.exists(CK_CLS):
    raise SystemExit('Khong thay classifier tai %s -> phai chay o 8 truoc.' % CK_CLS)

# ---- nap 2 model (dung lai neu o 9 da nap roi, do ton VRAM vo ich) ----
if 'model_det' not in globals():
    model_det = init_detector(os.path.join(WORK, 'cau_hinh_detector.py'), CKPT_DET, device='cuda:0')
    print('da nap detector:', os.path.basename(CKPT_DET))

ck = torch.load(CK_CLS, map_location='cuda')
# Ten lop lay THANG tu checkpoint (o 8 co luu kem 'ten_lop') — khong doan thu tu.
TEN_LOP_CLS = ck.get('ten_lop') or ['Flooded', 'Not-Flooded', 'Partially-Flooded']
if 'model_cls' not in globals():
    model_cls = convnext_tiny()
    model_cls.classifier[2] = nn.Linear(model_cls.classifier[2].in_features, len(TEN_LOP_CLS))
    model_cls.load_state_dict(ck['model'])
    model_cls = model_cls.cuda().eval()
    print('da nap classifier | lop:', TEN_LOP_CLS)
TF2 = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def phan_loai_crop(im, boxes):
    """boxes = pred_instances.bboxes cua mmdet = (x1, y1, x2, y2).
    Crop phai GIONG HET luc train (o 8): nha them 0.15 moi ben, lam tron +0.5."""
    if len(boxes) == 0:
        return []
    W, H = im.size
    batch = []
    for x1, y1, x2, y2 in boxes:
        m = 0.15
        bw, bh = x2 - x1, y2 - y1
        x0, y0 = max(0, int(x1 - bw * m)), max(0, int(y1 - bh * m))
        x3, y3 = min(W, int(x2 + bw * m + 0.5)), min(H, int(y2 + bh * m + 0.5))
        if x3 - x0 < 4 or y3 - y0 < 4:
            x0, y0, x3, y3 = 0, 0, W, H
        batch.append(TF2(im.crop((x0, y0, x3, y3)).resize((SIZE, SIZE), Image.BILINEAR)))
    with torch.no_grad():
        out = model_cls(torch.stack(batch).cuda()).argmax(1).cpu().numpy()
    return [TEN_LOP_CLS[i] for i in out]


def nhan_dien(im, nguong):
    if im is None:
        return None, '### Chua co anh — hay dua 1 anh flycam len truoc.'
    im = im.convert('RGB')
    # Ghi ra file tam roi dua DUONG DAN cho mmdet — y het cach o 9 da chay dung.
    # (Dua mang numpy thang vao co the lech kenh mau BGR/RGB, khong kiem chung duoc.)
    fd, p = tempfile.mkstemp(suffix='.png')
    os.close(fd)
    try:
        im.save(p)
        r = inference_detector(model_det, p)
    finally:
        try:
            os.remove(p)
        except OSError:
            pass

    b = r.pred_instances.bboxes.cpu().numpy()
    sc = r.pred_instances.scores.cpu().numpy()
    b = b[sc >= nguong]
    nhan = phan_loai_crop(im, b)
    dem = collections.Counter(nhan)

    dr = ImageDraw.Draw(im)
    for (x1, y1, x2, y2), n in zip(b, nhan):
        dr.rectangle([x1, y1, x2, y2], outline=MAU.get(n, (255, 255, 255)), width=2)

    L = ['### Kết quả', '', '| Loại nhà | Số lượng |', '|---|---:|']
    for k in TEN_LOP_CLS:
        L.append('| %s | %d |' % (k, dem[k]))
    L.append('| **Tổng số nhà** | **%d** |' % len(b))
    L.append('')
    L.append('Ngưỡng tin cậy của detector: **%.2f**' % nguong)
    L.append('')
    L.append('Màu khung: **đỏ** = ngập · **cam** = ngập một phần · **xanh lá** = không ngập')
    return im, '\n'.join(L)


with gr.Blocks(title='Nhận diện nhà ngập') as app:
    gr.Markdown('# Nhận diện & đếm nhà ngập\n'
                'Cascade R-CNN (tìm nhà) → ConvNeXt-Tiny (phân loại ngập / không ngập / ngập một phần). '
                'Đưa ảnh flycam lên, kết quả hiện ngay.')
    with gr.Row():
        with gr.Column():
            anh_vao = gr.Image(type='pil', label='Ảnh flycam')
            nguong = gr.Slider(minimum=0.05, maximum=0.90, value=THR, step=0.05,
                               label='Ngưỡng tin cậy của detector (càng thấp càng bắt được nhiều nhà nhỏ)')
            nut = gr.Button('Nhận diện', variant='primary')
        with gr.Column():
            anh_ra = gr.Image(type='pil', label='Kết quả')
            ket_qua = gr.Markdown()
    nut.click(nhan_dien, [anh_vao, nguong], [anh_ra, ket_qua])
    anh_vao.change(nhan_dien, [anh_vao, nguong], [anh_ra, ket_qua])

print('Dang mo web app — cho ~10 giay de lay link...')
app.launch(share=True)


---
## Khi cần chạy lại / train lại
- **Đứt kết nối:** `Runtime > Run all` lại. Ô 6 sẽ tự resume từ checkpoint trên Drive.
- **Muốn train lại từ đầu:** vào `MyDrive/nha_ngap_v002_out/` xoá thư mục `work_det` (detector) hoặc `work_cls/convnext_tiny_3lop.pth` (classifier), rồi Run all.
- **Hết VRAM:** giảm `BATCH = 2` xuống `1` ở ô 6.

## Mở web app (đưa ảnh lên → xem kết quả)
- Ô 11 mở web app và in ra link dạng `https://xxxx.gradio.live` — mở link đó là dùng được (điện thoại cũng vào được).
- **Phiên mới KHÔNG cần train lại:** chỉ chạy ô 1 → 2 → 3 rồi chạy thẳng ô 11 (2 model nạp từ checkpoint trên Drive, không cần dataset).
- Link `gradio.live` sống khoảng 72 giờ và chỉ khi tab Colab còn mở → lúc bảo vệ nên mở sẵn trước.
- Thanh trượt "ngưỡng tin cậy": hạ xuống thì bắt được nhiều nhà nhỏ hơn (đúng yêu cầu đề tài) nhưng dễ thêm nhà giả.

## Ghi chú kỹ thuật (để bảo vệ đồ án)
- Detector học **1 lớp `house`** (gộp cả 3 loại nhà) → chỉ lo *tìm* nhà, kể cả nhà rất bé. Đếm nhà = đếm box của tầng này.
- Classifier học **3 lớp** trên đúng box nhà → chỉ lo *phân loại* ngập / không / một phần.
- `bbox_mAP_s` ở ô 7 là chỉ số quan trọng nhất cho yêu cầu "đếm cả nhà nhỏ, nhà ở xa".
- Toàn bộ nhãn là nhãn có sẵn của dataset, notebook **không tự gán/sửa nhãn nào**.
